<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-05/blob/main/hw_5_rdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 5 — CTE, Recursive CTE and Materialized Views



## Task 1: Setup and Data Loading

In [2]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.6 MB/s eta 0:00:00


In [3]:
!pip install -q psycopg2-binary sqlalchemy pandas kagglehub

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 44.3 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [4]:
# Dataset import
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [5]:
import os

sorted(os.listdir(dataset_dir))

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [6]:
files = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
]

for file_name in files:
    df_tmp = pd.read_csv(os.path.join(dataset_dir, file_name))
    print(f"\n{file_name}")
    print("shape:", df_tmp.shape)
    print("columns:", df_tmp.columns.tolist())


olist_customers_dataset.csv
shape: (99441, 5)
columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_orders_dataset.csv
shape: (99441, 8)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items_dataset.csv
shape: (112650, 7)
columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_products_dataset.csv
shape: (32951, 9)
columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

olist_sellers_dataset.csv
shape: (3095, 4)
columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [7]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )

    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


### 1.2 Create Typed Tables with Constraints

In [8]:
with engine.begin() as conn:
    conn.execute(text("""

      DROP TABLE IF EXISTS olist_order_items CASCADE;
      DROP TABLE IF EXISTS olist_orders      CASCADE;
      DROP TABLE IF EXISTS olist_customers   CASCADE;
      DROP TABLE IF EXISTS olist_products    CASCADE;
      DROP TABLE IF EXISTS olist_sellers     CASCADE;

      CREATE TABLE olist_customers (
          customer_id              TEXT PRIMARY KEY,
          customer_unique_id       TEXT NOT NULL,
          customer_zip_code_prefix INTEGER,
          customer_city            TEXT,
          customer_state           CHAR(2) NOT NULL
      );

      CREATE TABLE olist_orders (
          order_id                      TEXT PRIMARY KEY,
          customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
          order_status                  TEXT NOT NULL,
          order_purchase_timestamp      TIMESTAMP NOT NULL,
          order_approved_at             TIMESTAMP,
          order_delivered_carrier_date  TIMESTAMP,
          order_delivered_customer_date TIMESTAMP,
          order_estimated_delivery_date TIMESTAMP,
          CHECK (order_status IN (
              'created', 'approved', 'invoiced', 'processing',
              'shipped', 'delivered', 'unavailable', 'canceled'
          ))
      );

      CREATE TABLE olist_products (
          product_id                 TEXT PRIMARY KEY,
          product_category_name      TEXT,
          product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
          product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
          product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
          product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
          product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
          product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
          product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
      );

      CREATE TABLE olist_sellers (
          seller_id              TEXT PRIMARY KEY,
          seller_zip_code_prefix INTEGER,
          seller_city            TEXT,
          seller_state           CHAR(2) NOT NULL
      );

      CREATE TABLE olist_order_items (
          order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
          order_item_id       INTEGER NOT NULL,
          product_id          TEXT REFERENCES olist_products(product_id),
          seller_id           TEXT REFERENCES olist_sellers(seller_id),
          shipping_limit_date TIMESTAMP,
          price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
          freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
          PRIMARY KEY (order_id, order_item_id)
      );

    """))
print("Successfully created typed tables.")

Successfully created typed tables.


### 1.3 Load Data from Raw to Typed Tables

In [9]:
with engine.begin() as conn:
    conn.execute(text("""

      INSERT INTO olist_customers
      SELECT
          customer_id,
          customer_unique_id,
          customer_zip_code_prefix::INTEGER,
          customer_city,
          customer_state::CHAR(2)
      FROM olist_customers_raw;

      INSERT INTO olist_orders
      SELECT
          order_id,
          customer_id,
          order_status,
          order_purchase_timestamp::TIMESTAMP,
          order_approved_at::TIMESTAMP,
          order_delivered_carrier_date::TIMESTAMP,
          order_delivered_customer_date::TIMESTAMP,
          order_estimated_delivery_date::TIMESTAMP
      FROM olist_orders_raw;

      INSERT INTO olist_products
      SELECT
          product_id,
          product_category_name,
          product_name_lenght::INTEGER,
          product_description_lenght::INTEGER,
          product_photos_qty::INTEGER,
          product_weight_g::INTEGER,
          product_length_cm::INTEGER,
          product_height_cm::INTEGER,
          product_width_cm::INTEGER
      FROM olist_products_raw;

      INSERT INTO olist_sellers
      SELECT
          seller_id,
          seller_zip_code_prefix::INTEGER,
          seller_city,
          seller_state::CHAR(2)
      FROM olist_sellers_raw;

      INSERT INTO olist_order_items
      SELECT
          order_id,
          order_item_id::INTEGER,
          product_id,
          seller_id,
          shipping_limit_date::TIMESTAMP,
          price::NUMERIC(12, 2),
          freight_value::NUMERIC(12, 2)
      FROM olist_order_items_raw;

  """))
print("Raw data successfully loaded into typed tables.")


Raw data successfully loaded into typed tables.


### 1.4 Validate Typed Table Row Counts

In [10]:
pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers

    UNION ALL

    SELECT 'orders', COUNT(*)
    FROM olist_orders

    UNION ALL

    SELECT 'order_items', COUNT(*)
    FROM olist_order_items

    UNION ALL

    SELECT 'products', COUNT(*)
    FROM olist_products

    UNION ALL

    SELECT 'sellers', COUNT(*)
    FROM olist_sellers

    ORDER BY table_name;
""", engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


## Task 2: CTE-chain feature pipeline

In [25]:
pd.read_sql("""

    WITH
    reference_date AS (
        -- Stage 0: stable reference date for historical dataset
        SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),

    stage1_orders_total AS (
        -- Stage 1: orders + order_total + status
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,
            COALESCE(SUM(
                COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
            ), 0)::NUMERIC(12, 2) AS order_total
        FROM olist_orders AS o
        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage2_rfm AS (
      -- Stage 2: RFM features per customer
      SELECT
          s.customer_id,
          (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
          COUNT(*) AS frequency,
          SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
          MIN(s.order_date) AS first_order_date
      FROM stage1_orders_total AS s
      CROSS JOIN reference_date AS rd
      GROUP BY
          s.customer_id,
          rd.as_of_date
    ),

    stage3_cohort AS (
      -- Stage 3: cohort quarter from first order date
      SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
      FROM stage2_rfm AS r
    ),

    stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
      SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
    )
Select *
From stage4_rolling_aov
Limit 5;

""", engine)

,customer_id,order_id,order_ts,order_total,avg_order_value_30d,rn_recent
0,00012a2ce6f8dcda20d059ce98491703,5f79b5b0931d63f1a42989eb65b9da6e,2017-11-14 16:08:26,114.74,114.74,1
1,000161a058600d5901f007fab4c27140,a44895d095d7e0702b6a162fa2dbeced,2017-07-16 09:40:32,67.41,67.41,1
2,0001fd6190edaaf884bcaf3d49edf079,316a104623542e4d75189bb372bc5f8d,2017-02-28 11:06:43,195.42,195.42,1
3,0002414f95344307404f0ace7a26f1d5,5825ce2e88d5346438686b0bba99e5ee,2017-08-16 13:09:20,179.35,179.35,1
4,000379cdec625522490c315e70c7a9fb,0ab7fb08086d4af9141453c91878ed7a,2018-04-02 13:42:17,107.01,107.01,1
